In [ ]:
"""
Orchestrator Training Data Pipeline
====================================

Converts A2A discovery phase call logs into structured training data for
orchestrator configuration (Agno, Temporal, or custom solution).

After 3 months of A2A agent-to-agent calls, this pipeline analyzes patterns
to extract:
1. Actual workflow sequences that work
2. Agent performance profiles and bottlenecks
3. Error patterns and failure modes
4. Optimization opportunities
5. Orchestrator configuration ready for production

The key insight: Instead of designing orchestration theoretically,
we train it on 3 months of observed patterns.

Usage:
    analyzer = DiscoveryDataAnalyzer(call_logs)
    training_config = analyzer.generate_orchestrator_config()
    
    # Save for orchestrator deployment
    with open("orchestrator_training.json", "w") as f:
        json.dump(training_config, f, indent=2)
    
    # Deploy to Agno
    orchestrator = AgnoOrchestrator.from_training_config(training_config)
    orchestrator.start()
"""

import json
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict, Counter
import statistics


# ============================================================================
# PART 1: DATA MODELS FOR TRAINING CONFIGURATION
# ============================================================================

@dataclass
class AgentPerformanceProfile:
    """Performance characteristics learned from A2A calls"""
    agent_id: str
    avg_latency_ms: float
    p50_latency_ms: float
    p95_latency_ms: float
    p99_latency_ms: float
    max_latency_ms: float
    success_rate: float
    error_rate: float
    timeout_rate: float
    calls_observed: int
    timeout_threshold_recommended_ms: int
    capacity_concurrent_calls: int
    capacity_throughput_per_minute: float
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class WorkflowStep:
    """Single step in an orchestrated workflow"""
    step: int
    agent: str
    operation: str
    depends_on: List[int] = field(default_factory=list)
    fan_out_to: List[int] = field(default_factory=list)
    timeout_ms: int = 5000
    required: bool = True
    fallback: Optional[str] = None
    retry_policy: str = "exponential_backoff"
    max_retries: int = 3
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class WorkflowObservations:
    """Observed behavior of a workflow in discovery phase"""
    frequency: int
    success_rate: float
    avg_total_latency_ms: float
    p95_total_latency_ms: float
    depth_limit_violations: int
    depth_violations_resolved_at_step: Optional[int] = None
    common_failures: List[Dict] = field(default_factory=list)
    time_period_days: int = 90
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class OptimizationInsight:
    """Recommended optimization based on observed patterns"""
    insight_id: str
    title: str
    description: str
    affected_workflows: List[str]
    latency_improvement_percent: Optional[float] = None
    throughput_improvement_percent: Optional[float] = None
    cost_savings_percent: Optional[float] = None
    implementation_effort: str = "medium"  # "trivial", "medium", "high"
    estimated_hours: int = 0
    recommended: bool = True
    priority: int = 1  # 1=critical, 5=nice-to-have
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class ErrorPattern:
    """Common error mode observed during discovery"""
    error_type: str
    frequency: int
    percentage_of_failures: float
    triggering_conditions: List[str]
    affected_workflows: List[str]
    affected_agents: List[str]
    typical_resolution: str
    prevention_strategy: str
    
    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class Workflow:
    """Complete workflow definition trained on discovery data"""
    workflow_id: str
    business_outcome: str
    description: str
    triggering_agents: List[str]
    sequence: List[WorkflowStep]
    parallelization: Dict = field(default_factory=dict)
    observed: Optional[WorkflowObservations] = None
    confidence_score: float = 0.0
    
    def to_dict(self) -> Dict:
        data = asdict(self)
        if self.observed:
            data['observed'] = self.observed.to_dict()
        return data


# ============================================================================
# PART 2: DISCOVERY DATA ANALYZER
# ============================================================================

class DiscoveryDataAnalyzer:
    """
    Analyze A2A discovery phase logs and produce orchestrator training data.
    
    Input: 3 months of A2A call logs
    Output: Orchestrator configuration ready for production deployment
    """
    
    def __init__(self, call_logs: List[Dict]):
        """
        Initialize analyzer with call logs from discovery phase
        
        Args:
            call_logs: List of A2A call log entries with format:
                {
                    "timestamp": "ISO timestamp",
                    "caller": "agent-id",
                    "goal": "what was requested",
                    "target": "agent-id or None",
                    "depth": 0,
                    "status": "success|error|depth_limit_exceeded|timeout",
                    "latency_ms": float,
                    "error": dict or None
                }
        """
        self.logs = call_logs
        self.workflows_observed: Dict[str, List[Dict]] = defaultdict(list)
        self.agent_metrics: Dict[str, Dict] = {}
        self.error_patterns: List[Dict] = []
        self._analyze()
    
    def _analyze(self) -> None:
        """Run all analysis passes over the logs"""
        self._build_workflow_graph()
        self._calculate_agent_metrics()
        self._detect_error_patterns()
    
    def _build_workflow_graph(self) -> None:
        """
        Build a call graph from logs to identify workflow patterns.
        
        Key insight: A workflow is a sequence of calls where:
        - Agent A calls Agent B (depth 0)
        - Agent B calls Agent C (depth 1)
        - This forms a chain: A → B → C
        
        We track:
        - How often each sequence occurs
        - Success/failure rates
        - Where it hits depth limits
        """
        
        # Group by trace_id to reconstruct full request chains
        traces: Dict[str, List[Dict]] = defaultdict(list)
        
        for log in self.logs:
            # Note: A2A logs should include trace_id for distributed tracing
            trace_id = log.get('trace_id', 'unknown')
            traces[trace_id].append(log)
        
        # Extract workflows from traces
        for trace_id, trace_logs in traces.items():
            # Sort by timestamp to preserve order
            sorted_logs = sorted(trace_logs, key=lambda x: x.get('timestamp', ''))
            
            # Find the sequence of agent calls
            for i, log in enumerate(sorted_logs):
                if log['status'] == 'success':
                    # This is a successful call
                    workflow_key = f"{log['caller']} → {log['target']}"
                    self.workflows_observed[workflow_key].append({
                        'trace_id': trace_id,
                        'log': log,
                        'next_call': sorted_logs[i+1] if i+1 < len(sorted_logs) else None
                    })
    
    def _calculate_agent_metrics(self) -> None:
        """
        Calculate performance metrics for each agent.
        
        Metrics:
        - Latency: avg, p50, p95, p99
        - Success rate, error rate, timeout rate
        - Recommended timeouts
        - Capacity estimates
        """
        
        agent_calls: Dict[str, List[float]] = defaultdict(list)
        agent_stats: Dict[str, Dict] = defaultdict(lambda: {
            'success': 0,
            'error': 0,
            'timeout': 0,
            'depth_limit': 0,
            'total': 0
        })
        
        for log in self.logs:
            agent_id = log.get('target')
            if not agent_id:
                continue
            
            # Collect latency
            if log.get('latency_ms'):
                agent_calls[agent_id].append(log['latency_ms'])
            
            # Count outcomes
            status = log.get('status', 'unknown')
            agent_stats[agent_id]['total'] += 1
            
            if status == 'success':
                agent_stats[agent_id]['success'] += 1
            elif status == 'timeout':
                agent_stats[agent_id]['timeout'] += 1
            elif status == 'depth_limit_exceeded':
                agent_stats[agent_id]['depth_limit'] += 1
            else:
                agent_stats[agent_id]['error'] += 1
        
        # Build performance profiles
        for agent_id, latencies in agent_calls.items():
            if not latencies:
                continue
            
            latencies_sorted = sorted(latencies)
            stats = agent_stats[agent_id]
            total_calls = stats['total']
            
            # Calculate percentiles
            p50 = statistics.median(latencies)
            p95 = latencies_sorted[int(len(latencies_sorted) * 0.95)]
            p99 = latencies_sorted[int(len(latencies_sorted) * 0.99)]
            avg = statistics.mean(latencies)
            
            success_rate = stats['success'] / total_calls if total_calls > 0 else 0
            error_rate = stats['error'] / total_calls if total_calls > 0 else 0
            timeout_rate = stats['timeout'] / total_calls if total_calls > 0 else 0
            
            # Recommend timeout as p99 + 20% buffer
            recommended_timeout = int(p99 * 1.2)
            
            # Estimate capacity based on latency and success rate
            # (simplified: assume 1000ms budget per request)
            requests_per_minute = (60000 / (avg if avg > 0 else 1)) * success_rate
            concurrent_capacity = int(requests_per_minute / 10)  # Simplified estimate
            
            self.agent_metrics[agent_id] = AgentPerformanceProfile(
                agent_id=agent_id,
                avg_latency_ms=round(avg, 2),
                p50_latency_ms=round(p50, 2),
                p95_latency_ms=round(p95, 2),
                p99_latency_ms=round(p99, 2),
                max_latency_ms=round(max(latencies), 2),
                success_rate=round(success_rate, 3),
                error_rate=round(error_rate, 3),
                timeout_rate=round(timeout_rate, 3),
                calls_observed=total_calls,
                timeout_threshold_recommended_ms=recommended_timeout,
                capacity_concurrent_calls=max(1, concurrent_capacity),
                capacity_throughput_per_minute=round(requests_per_minute, 1)
            )
    
    def _detect_error_patterns(self) -> None:
        """
        Find common error modes and their causes.
        
        Error patterns tell us:
        - What kinds of failures happen
        - Under what conditions
        - How to prevent/handle them
        """
        
        error_counts: Counter = Counter()
        error_details: Dict[str, List[Dict]] = defaultdict(list)
        
        for log in self.logs:
            if log.get('status') != 'success':
                error_type = log.get('status', 'unknown')
                error_counts[error_type] += 1
                
                error_details[error_type].append({
                    'caller': log.get('caller'),
                    'target': log.get('target'),
                    'goal': log.get('goal'),
                    'error': log.get('error'),
                    'timestamp': log.get('timestamp')
                })
        
        total_calls = len(self.logs)
        
        # Build error patterns
        for error_type, count in error_counts.most_common():
            details = error_details[error_type]
            
            # Analyze error details for patterns
            affected_agents = set()
            affected_workflows = set()
            
            for detail in details:
                if detail['target']:
                    affected_agents.add(detail['target'])
                if detail['caller']:
                    affected_workflows.add(f"{detail['caller']} → {detail['target']}")
            
            # Determine prevention strategy based on error type
            prevention_strategies = {
                'timeout': 'Increase timeout threshold and/or add circuit breaker',
                'depth_limit_exceeded': 'Implement workflow orchestrator to handle cascades',
                'error': 'Add retry logic with exponential backoff',
                'not_found': 'Improve agent discovery or add fallback agents'
            }
            
            pattern = ErrorPattern(
                error_type=error_type,
                frequency=count,
                percentage_of_failures=round(count / total_calls * 100, 1),
                triggering_conditions=[
                    "See affected_agents and affected_workflows below"
                ],
                affected_workflows=list(affected_workflows),
                affected_agents=list(affected_agents),
                typical_resolution=f"Handle {error_type} in orchestration logic",
                prevention_strategy=prevention_strategies.get(error_type, "Add specific handling")
            )
            
            self.error_patterns.append(pattern)
    
    def extract_workflows(self) -> List[Workflow]:
        """
        Extract complete workflow definitions from observed patterns.
        
        A workflow is:
        - A sequence of agent calls that solves a business problem
        - Observed at least once in the logs
        - Either succeeds consistently or has known failure modes
        - Can be orchestrated with proper timeout/retry/fallback logic
        """
        
        workflows: List[Workflow] = []
        
        # Define known workflows and verify them against observations
        known_workflows = {
            "evaluate_funding_opportunity": {
                "business_outcome": "Determine if we should pursue a specific funding opportunity",
                "description": "Cross-check local capacity, investor fit, and market opportunity",
                "triggering_agents": ["funding-strategy-agent", "field-operations-agent"],
                "sequence": [
                    WorkflowStep(
                        step=1,
                        agent="field-operations-agent",
                        operation="assess_local_capacity_and_demand",
                        timeout_ms=300,
                        required=True,
                        fan_out_to=[2, 3]
                    ),
                    WorkflowStep(
                        step=2,
                        agent="fundraising-agent",
                        operation="evaluate_investor_fit_for_opportunity",
                        depends_on=[1],
                        timeout_ms=200,
                        required=True,
                        fallback="use_historical_investor_patterns"
                    ),
                    WorkflowStep(
                        step=3,
                        agent="business-development-agent",
                        operation="analyze_market_fit_and_competition",
                        depends_on=[1],
                        timeout_ms=500,
                        required=True,
                        fallback="use_cached_rfp_data"
                    )
                ],
                "parallelization": {
                    "steps_2_and_3_run_in_parallel": True,
                    "reason": "Both depend only on step 1 output"
                }
            },
            
            "identify_funding_gap": {
                "business_outcome": "Find mismatch between local demand and available funding",
                "description": "Identify unmet funding needs in a country/sector",
                "triggering_agents": ["field-operations-agent"],
                "sequence": [
                    WorkflowStep(
                        step=1,
                        agent="field-operations-agent",
                        operation="assess_local_demand_and_gaps",
                        timeout_ms=300,
                        required=True,
                        fan_out_to=[2]
                    ),
                    WorkflowStep(
                        step=2,
                        agent="business-development-agent",
                        operation="get_available_competitive_funding",
                        depends_on=[1],
                        timeout_ms=500,
                        required=True,
                        fallback="use_historical_funding_data"
                    )
                ],
                "parallelization": {}
            },
            
            "assess_investor_suitability": {
                "business_outcome": "Determine if an investor is suitable for a specific opportunity",
                "description": "Match investor profile against opportunity characteristics",
                "triggering_agents": ["funding-strategy-agent"],
                "sequence": [
                    WorkflowStep(
                        step=1,
                        agent="fundraising-agent",
                        operation="get_investor_profile_and_interests",
                        timeout_ms=200,
                        required=True
                    )
                ],
                "parallelization": {}
            }
        }
        
        # Build workflows with observations
        for workflow_id, template in known_workflows.items():
            workflow_key = self._find_workflow_in_observations(workflow_id)
            observations = None
            confidence = 0.0
            
            if workflow_key:
                observations_data = self._calculate_workflow_observations(workflow_key)
                observations = WorkflowObservations(**observations_data)
                # Confidence based on frequency and success rate
                confidence = min(observations.success_rate, 
                               min(observations.frequency / 100, 1.0))
            
            workflow = Workflow(
                workflow_id=workflow_id,
                business_outcome=template['business_outcome'],
                description=template['description'],
                triggering_agents=template['triggering_agents'],
                sequence=template['sequence'],
                parallelization=template['parallelization'],
                observed=observations,
                confidence_score=round(confidence, 2)
            )
            
            workflows.append(workflow)
        
        return workflows
    
    def _find_workflow_in_observations(self, workflow_id: str) -> Optional[str]:
        """Find a workflow key in observed patterns"""
        # Simplified: just return the first matching key
        for key in self.workflows_observed.keys():
            if workflow_id.replace('_', ' ') in key.lower():
                return key
        return None
    
    def _calculate_workflow_observations(self, workflow_key: str) -> Dict:
        """Calculate observations for a workflow"""
        observations = self.workflows_observed.get(workflow_key, [])
        
        if not observations:
            return {
                'frequency': 0,
                'success_rate': 0.0,
                'avg_total_latency_ms': 0.0,
                'p95_total_latency_ms': 0.0,
                'depth_limit_violations': 0
            }
        
        latencies = [obs['log'].get('latency_ms', 0) for obs in observations 
                    if obs['log'].get('latency_ms')]
        
        return {
            'frequency': len(observations),
            'success_rate': sum(1 for obs in observations if obs['log']['status'] == 'success') / len(observations),
            'avg_total_latency_ms': round(statistics.mean(latencies), 2) if latencies else 0,
            'p95_total_latency_ms': round(sorted(latencies)[int(len(latencies)*0.95)], 2) if latencies else 0,
            'depth_limit_violations': sum(1 for obs in observations 
                                         if obs['log']['status'] == 'depth_limit_exceeded')
        }
    
    def identify_required_cascades(self) -> List[Dict]:
        """
        Find workflows that MUST be orchestrated because they hit depth limits.
        
        Key insight: If agents want to call A → B → C but get blocked by depth limit,
        the orchestrator MUST support this sequence to unblock the workflow.
        """
        
        required_cascades = []
        
        for log in self.logs:
            if log.get('status') == 'depth_limit_exceeded':
                # This is a workflow that MUST be supported
                cascade = {
                    'caller': log.get('caller'),
                    'attempted_target': log.get('target'),
                    'goal': log.get('goal'),
                    'current_depth': log.get('depth'),
                    'max_depth': log.get('max_depth', 2),
                    'frequency': 0,  # Will be incremented below
                    'blocking_agents': []
                }
                
                # Check if we already have this cascade
                found = False
                for existing in required_cascades:
                    if (existing['caller'] == cascade['caller'] and 
                        existing['attempted_target'] == cascade['attempted_target']):
                        existing['frequency'] += 1
                        found = True
                        break
                
                if not found:
                    cascade['frequency'] = 1
                    required_cascades.append(cascade)
        
        # Sort by frequency (most important first)
        return sorted(required_cascades, key=lambda x: x['frequency'], reverse=True)
    
    def recommend_optimizations(self) -> List[OptimizationInsight]:
        """
        Recommend optimizations based on observed patterns.
        
        Optimizations:
        - Parallelization opportunities
        - Caching strategies
        - Timeout tuning
        - Agent load balancing
        - Fallback strategies
        """
        
        optimizations: List[OptimizationInsight] = []
        
        # Optimization 1: Parallelization
        for workflow in self.extract_workflows():
            if workflow.observed and workflow.observed.frequency > 10:
                # Check if steps can be parallelized
                steps_by_dependencies = defaultdict(list)
                for step in workflow.sequence:
                    key = tuple(sorted(step.depends_on))
                    steps_by_dependencies[key].append(step.step)
                
                # If 2+ steps have same dependencies, they can run in parallel
                parallel_opportunities = [steps for steps in steps_by_dependencies.values() 
                                         if len(steps) > 1]
                
                if parallel_opportunities:
                    # Calculate potential improvement
                    # Simplified: assume linear sum of latencies currently
                    agents = [self.agent_metrics.get(f"{workflow.workflow_id}-agent") 
                             for _ in parallel_opportunities]
                    
                    optimization = OptimizationInsight(
                        insight_id=f"parallelize_{workflow.workflow_id}",
                        title=f"Parallelize steps in {workflow.workflow_id}",
                        description=f"Run steps {parallel_opportunities[0]} in parallel instead of sequential",
                        affected_workflows=[workflow.workflow_id],
                        latency_improvement_percent=25,  # Rough estimate
                        implementation_effort="trivial",
                        estimated_hours=2,
                        recommended=True,
                        priority=1
                    )
                    optimizations.append(optimization)
        
        # Optimization 2: Caching for slow agents
        for agent_id, profile in self.agent_metrics.items():
            if profile.avg_latency_ms > 100 and profile.success_rate > 0.9:
                optimization = OptimizationInsight(
                    insight_id=f"cache_{agent_id}",
                    title=f"Add caching layer for {agent_id}",
                    description=f"{agent_id} has {profile.avg_latency_ms}ms avg latency. "
                               f"Caching could improve by 60-80%.",
                    affected_workflows=["all"],
                    latency_improvement_percent=70,
                    implementation_effort="medium",
                    estimated_hours=8,
                    recommended=True,
                    priority=2
                )
                optimizations.append(optimization)
        
        # Optimization 3: Timeout tuning
        for agent_id, profile in self.agent_metrics.items():
            if profile.timeout_rate > 0.01:  # >1% timeout rate
                optimization = OptimizationInsight(
                    insight_id=f"tune_timeout_{agent_id}",
                    title=f"Increase timeout for {agent_id}",
                    description=f"{agent_id} has {profile.timeout_rate*100:.1f}% timeout rate. "
                               f"Current recommended: {profile.timeout_threshold_recommended_ms}ms",
                    affected_workflows=["all"],
                    latency_improvement_percent=None,
                    implementation_effort="trivial",
                    estimated_hours=0.5,
                    recommended=True,
                    priority=1
                )
                optimizations.append(optimization)
        
        # Optimization 4: Fallback strategies
        required_cascades = self.identify_required_cascades()
        if required_cascades:
            optimization = OptimizationInsight(
                insight_id="implement_fallbacks",
                title="Implement smart fallback strategies",
                description=f"{len(required_cascades)} cascade workflows blocked by depth limits. "
                          f"Add fallback agents or cache for {required_cascades[0]['attempted_target']}",
                affected_workflows=[cascade['goal'] for cascade in required_cascades[:3]],
                latency_improvement_percent=None,
                implementation_effort="medium",
                estimated_hours=6,
                recommended=True,
                priority=1
            )
            optimizations.append(optimization)
        
        return optimizations
    
    def generate_orchestrator_config(self) -> Dict:
        """
        Generate complete orchestrator configuration file.
        
        Output format suitable for:
        - Agno orchestrator
        - Temporal workflows
        - Custom orchestration engine
        """
        
        return {
            "metadata": {
                "generated_at": datetime.utcnow().isoformat(),
                "training_phase_duration_days": 90,
                "total_calls_analyzed": len(self.logs),
                "discovery_system": "A2A Protocol",
                "target_orchestrator": "Agno (or custom)"
            },
            
            "workflows": [
                workflow.to_dict() for workflow in self.extract_workflows()
            ],
            
            "agent_performance_profiles": [
                profile.to_dict() for profile in self.agent_metrics.values()
            ],
            
            "required_cascades": self.identify_required_cascades(),
            
            "error_patterns": [
                asdict(pattern) for pattern in self.error_patterns
            ],
            
            "optimization_insights": [
                optimization.to_dict() for optimization in self.recommend_optimizations()
            ],
            
            "deployment_recommendations": {
                "immediate_actions": [
                    "Deploy orchestrator with workflows having >90% confidence",
                    "Set timeouts based on agent_performance_profiles",
                    "Implement fallback strategies for common error patterns",
                    "Enable distributed tracing with trace_id propagation"
                ],
                "phase_2_improvements": [
                    "Implement recommended optimizations (parallel execution, caching)",
                    "Monitor orchestrator performance against discovery phase baselines",
                    "Adjust timeouts and retry policies based on real usage"
                ],
                "monitoring_metrics": [
                    "Workflow success rate (target: >95%)",
                    "End-to-end latency (target: <500ms p95)",
                    "Agent response times (compare to discovery phase baselines)",
                    "Cascade depth violations (target: 0 in production)"
                ]
            }
        }


# ============================================================================
# PART 3: ORCHESTRATOR INTEGRATION EXAMPLES
# ============================================================================

class OrchestratorDeployer:
    """
    Deploy orchestrator using training data from discovery phase.
    
    This is a template showing how to use the training config with
    different orchestrator frameworks.
    """
    
    def __init__(self, training_config: Dict):
        self.config = training_config
    
    def deploy_to_agno(self):
        """
        Example: Deploy to Agno orchestrator
        
        Pseudocode (actual implementation depends on Agno API):
        """
        
        example_code = """
        from agno_client import AgnoOrchestrator
        
        # Load training configuration
        training_config = load_json("orchestrator_training.json")
        
        # Create orchestrator
        orchestrator = AgnoOrchestrator(project_id="funder-intelligence")
        
        # Register agents from performance profiles
        for agent_profile in training_config['agent_performance_profiles']:
            orchestrator.register_agent(
                agent_id=agent_profile['agent_id'],
                timeout_ms=agent_profile['timeout_threshold_recommended_ms'],
                max_concurrent=agent_profile['capacity_concurrent_calls'],
                circuit_breaker_threshold=1 - agent_profile['success_rate']
            )
        
        # Define workflows from training data
        for workflow in training_config['workflows']:
            orchestrator.define_workflow(
                workflow_id=workflow['workflow_id'],
                description=workflow['business_outcome'],
                steps=[
                    {
                        'step': step['step'],
                        'agent': step['agent'],
                        'operation': step['operation'],
                        'depends_on': step['depends_on'],
                        'timeout_ms': step['timeout_ms'],
                        'fallback': step['fallback'],
                        'retry_policy': step['retry_policy']
                    }
                    for step in workflow['sequence']
                ],
                parallelization=workflow['parallelization'],
                success_rate_target=workflow.get('observed', {}).get('success_rate', 0.95)
            )
        
        # Apply optimizations
        for optimization in training_config['optimization_insights']:
            if optimization['recommended']:
                orchestrator.apply_optimization(optimization)
        
        # Start orchestrator
        orchestrator.start()
        print(f"Orchestrator deployed with {len(training_config['workflows'])} workflows")
        """
        
        return example_code
    
    def deploy_to_temporal(self):
        """Example: Deploy to Temporal workflow engine"""
        
        example_code = """
        from temporalio.client import Client
        from temporalio.worker import Worker
        
        # Load training configuration
        training_config = load_json("orchestrator_training.json")
        
        # Register Temporal workflows for each discovered workflow
        for workflow in training_config['workflows']:
            @dataclass
            class WorkflowInput:
                country: str
                sector: str
                investor_id: str
            
            class FunderIntelligenceWorkflow:
                @workflow.run
                async def main(self, input: WorkflowInput):
                    # Execute steps from training data
                    for step in workflow['sequence']:
                        result = await workflow.execute_activity(
                            activity_name=step['operation'],
                            args=(step['agent'], step['operation'], input),
                            start_to_close_timeout=timedelta(
                                milliseconds=step['timeout_ms']
                            ),
                            retry_policy=RetryPolicy(
                                max_attempts=step.get('max_retries', 3)
                            )
                        )
                    
                    return result
        """
        
        return example_code


# ============================================================================
# PART 4: EXAMPLE USAGE AND DATA GENERATION
# ============================================================================

def generate_synthetic_discovery_logs() -> List[Dict]:
    """
    Generate synthetic A2A logs for demonstration.
    
    In production, these come from 3 months of real A2A calls.
    """
    
    logs = []
    agents = [
        "fundraising-agent",
        "business-development-agent", 
        "field-operations-agent"
    ]
    
    # Simulate 3 months of calls (sample: ~1000 calls per month = 3000 total)
    for i in range(3000):
        caller = agents[i % len(agents)]
        target = agents[(i + 1) % len(agents)]
        depth = (i // 100) % 3  # Simulate varying depths
        
        # Most calls succeed, some timeout/error
        status_rand = (i * 17) % 100  # Pseudo-random
        if status_rand > 95:
            status = "timeout"
            latency = 5000 + (status_rand * 10)
        elif status_rand > 85:
            status = "depth_limit_exceeded"
            latency = 10
        else:
            status = "success"
            # Agent-specific latency profiles
            if target == "business-development-agent":
                latency = 100 + (i % 200)
            elif target == "fundraising-agent":
                latency = 40 + (i % 80)
            else:
                latency = 70 + (i % 120)
        
        logs.append({
            "timestamp": (datetime.utcnow() - timedelta(days=90) + timedelta(minutes=i)).isoformat(),
            "trace_id": f"trace-{i // 10}",
            "caller": caller,
            "target": target,
            "goal": f"goal_{i % 10}",
            "depth": depth,
            "status": status,
            "latency_ms": latency,
            "error": {"code": status} if status != "success" else None
        })
    
    return logs


def main():
    """Example: Generate training config from discovery logs"""
    
    print("=" * 80)
    print("ORCHESTRATOR TRAINING PIPELINE - EXAMPLE")
    print("=" * 80)
    print()


In [ ]:
# Load or generate discovery logs
    print("Loading A2A discovery phase logs...")
    logs = generate_synthetic_discovery_logs()
    print(f"Loaded {len(logs)} call logs")
    print()


In [ ]:
# Analyze logs
    print("Analyzing discovery phase data...")
    analyzer = DiscoveryDataAnalyzer(logs)
    print(f"Found {len(analyzer.agent_metrics)} agents")
    print(f"Found {len(analyzer.error_patterns)} error patterns")
    print()


In [ ]:
# Generate orchestrator config
    print("Generating orchestrator training configuration...")
    training_config = analyzer.generate_orchestrator_config()
    
    # Display results
    print("\n" + "=" * 80)
    print("AGENT PERFORMANCE PROFILES")
    print("=" * 80)
    for agent in training_config['agent_performance_profiles']:
        print(f"\n{agent['agent_id']}:")
        print(f"  Avg Latency: {agent['avg_latency_ms']}ms")
        print(f"  P95 Latency: {agent['p95_latency_ms']}ms")
        print(f"  Success Rate: {agent['success_rate']*100:.1f}%")
        print(f"  Recommended Timeout: {agent['timeout_threshold_recommended_ms']}ms")
        print(f"  Capacity: {agent['capacity_concurrent_calls']} concurrent")


In [ ]:
print("\n" + "=" * 80)
    print("WORKFLOWS")
    print("=" * 80)
    for workflow in training_config['workflows']:
        print(f"\n{workflow['workflow_id']}:")
        print(f"  Business Outcome: {workflow['business_outcome']}")
        if workflow.get('observed'):
            obs = workflow['observed']
            print(f"  Observed Frequency: {obs['frequency']} times")
            print(f"  Success Rate: {obs['success_rate']*100:.1f}%")
            print(f"  Depth Violations: {obs['depth_limit_violations']}")


In [ ]:
print("\n" + "=" * 80)
    print("REQUIRED CASCADES")
    print("=" * 80)
    for cascade in training_config['required_cascades'][:5]:
        print(f"\n{cascade['caller']} → {cascade['attempted_target']}:")
        print(f"  Frequency (blocked): {cascade['frequency']}")
        print(f"  Goal: {cascade['goal']}")


In [ ]:
print("\n" + "=" * 80)
    print("OPTIMIZATION INSIGHTS")
    print("=" * 80)
    for opt in training_config['optimization_insights'][:5]:
        print(f"\n{opt['insight_id']}:")
        print(f"  {opt['title']}")
        print(f"  Effort: {opt['implementation_effort']}")
        if opt.get('latency_improvement_percent'):
            print(f"  Improvement: {opt['latency_improvement_percent']}%")


In [ ]:
# Save to file
    output_file = "/mnt/user-data/outputs/orchestrator_training.json"
    print(f"\n\nSaving training configuration to {output_file}...")
    with open(output_file, "w") as f:
        json.dump(training_config, f, indent=2, default=str)
    
    print("✓ Training configuration saved")
    print("\nNext steps:")
    print("1. Review the training configuration")
    print("2. Deploy to Agno with the configuration")
    print("3. Monitor orchestrator performance against discovery phase baselines")
    print("4. Apply recommended optimizations in phase 2")


In [ ]:
if __name__ == "__main__":
    main()
